# Advanced SQL Concepts (COUNT DISTINCT and Subqueries)

## Introduction: Extending Your SQL Toolkit

Throughout Unit 3, you learned the foundational clauses of Structured Query Language: `SELECT`, `FROM`, `JOIN`, `WHERE`, `GROUP BY`, `HAVING`, and `ORDER BY`. With these core tools, you can answer the vast majority of enterprise business questions—retrieving raw records, linking relational tables, filtering subsets, creating calculated transformations, and aggregating transactions into summary reports.

However, real-world corporate auditing and financial analytics frequently present questions that fall outside standard row-by-row filtering or routine category grouping. For example:
* How many foreign countries did government executives visit on official travel this year?
* Which corporate audit clients paid above-average audit fees relative to the industry benchmark?
* Which credit card charges occurred in the last 30 days of the fiscal year, when the exact ending date of the database is unknown in advance?

Answering these questions requires **intermediate SQL techniques**. While this supplemental chapter introduces concepts beyond the basic required course assignments, mastering these techniques will solidify your understanding of SQL execution logic, sharpen your forensic auditing capabilities, and prepare you for advanced modeling in tools like Microsoft Excel, Power BI, and Tableau.

---

## Topic 1: Counting Unique Entities with `COUNT(DISTINCT)`

Previously, you learned that `COUNT(*)` returns the total number of rows in a table, while `COUNT(ColumnName)` counts the number of non-null values in a specific column. However, both of these functions count every instance of a value, **including duplicates.**

In financial auditing and business analytics, you frequently need to know the number of **unique** or **distinct** entities rather than the total count of transactions.

### The Problem: Duplicate Transaction Records

Consider a government agency tracking travel expenses for public officials. If an executive takes ten separate business trips to London, England during the fiscal year, a standard `COUNT(DestinationCountry)` query will count `'United Kingdom'` ten separate times. 

If a legislative committee asks, *"To how many unique foreign countries did our executives travel?"*, simply counting transaction rows will drastically inflate the result.

### The Solution: `COUNT(DISTINCT ColumnName)`

To count only the unique values in a column, embed the **`DISTINCT`** keyword directly inside the `COUNT()` aggregate function:

```sql
COUNT(DISTINCT ColumnName)
```

SQL processes `COUNT(DISTINCT)` in two sequential phases:
1. **Scan and Deduplicate:** It inspects the specified column across all filtered rows and eliminates all duplicate values in memory.
2. **Count Scalar:** It counts the remaining unique entries and returns a single integer.

```{note}
**`SELECT DISTINCT` vs. `COUNT(DISTINCT)`**
* **`SELECT DISTINCT Country FROM Trips`** returns a **multi-row table** listing the actual names of the unique countries (e.g., `'France'`, `'Germany'`, `'Japan'`).
* **`SELECT COUNT(DISTINCT Country) FROM Trips`** returns a **single number** representing how many unique countries exist in the dataset (e.g., `3`).
```

### Case Study: Canadian Government Employee Travel (`Oh Canada` Database)

Consider an audit of public sector travel expenses using the `Oh Canada` database (`Trips` table), which records travel details for Canadian government employees. A parliamentary minister requests a summary metric: **How many unique travel destinations (countries) outside of Canada were visited by government personnel?**

To answer this question, we combine `WHERE` filtering with `COUNT(DISTINCT)`:

```sql
SELECT 
    COUNT(DestinationCountry) AS TotalForeignTrips,
    COUNT(DISTINCT DestinationCountry) AS UniqueForeignCountries
FROM Trips
WHERE DestinationCountry <> 'Canada'
```

*Expected Query Output:*
| TotalForeignTrips | UniqueForeignCountries |
| :--- | :--- |
| 1,420 | 48 |

*Analytical Takeaway:* While Canadian officials completed 1,420 international trips (`TotalForeignTrips`), those trips were concentrated across only 48 unique foreign nations (`UniqueForeignCountries`). Distinguishing total volume from unique reach is a vital auditing capability when evaluating travel policy compliance and geographic risk exposure.

---

## Topic 2: Comparing Rows to Summary Statistics (Scalar Subqueries)

A common task in financial analysis and forensic auditing is identifying **outliers** — records that deviate significantly from a population average. For example:
* Finding companies that pay **higher-than-average audit fees**.
* Flagging employees whose credit card purchases exceed the **departmental average expense**.
* Identifying products sold at **unusually low profit margins** compared to the portfolio average.

### The Precedence Trap: Why `WHERE Value > AVG(Value)` Fails

When asked to find companies with above-average audit fees, beginner analysts often write the following query:

```sql
-- CAUTION: THIS QUERY PRODUCES A SYNTAX ERROR!
SELECT CompanyName, AuditFees
FROM Top500
WHERE AuditFees > AVG(AuditFees)
```

If you execute this query, SQL Server immediately throws an error:  
`An aggregate may not appear in the WHERE clause unless it is in a subquery...`

**Why does this happen?**  
Recall the strict **SQL Clause Execution Sequence** established throughout this textbook:
1. `FROM` / `JOIN` (Identify source tables)
2. `WHERE` (Filter individual transaction rows)
3. `GROUP BY` (Collapse rows into categories)
4. `HAVING` (Filter aggregated groups)
5. `SELECT` (Compute output columns and aggregations)
6. `ORDER BY` (Sort final results)

The `WHERE` clause is evaluated at **Step 2**, row-by-row, *before* the database engine has evaluated the full table. Because `AVG(AuditFees)` requires scanning the entire table to compute the average, that benchmark value **does not yet exist** when the `WHERE` clause is filtering individual rows!

### The Solution: Scalar Subqueries

To solve this execution order dilemma, we use a **subquery** (also known as an *nested query* or *inner query*).

A **scalar subquery** is a complete `SELECT` statement enclosed in parentheses `()` that returns a **single value** (one row, one column). Because SQL evaluates expressions inside parentheses first, the database engine executes the inner subquery, calculates the scalar summary statistic (e.g., the overall average audit fee), and passes that exact numeric value directly into the outer query's `WHERE` clause!

### Subquery Syntax and Structure

```sql
SELECT Column1, Column2, ...
FROM TableName
WHERE BenchmarkColumn > (
    -- SCALAR SUBQUERY: Computes the single benchmark value first
    SELECT AVG(BenchmarkColumn)
    FROM TableName
)
```

### Case Study: Audit Fee Benchmarking (`Top500` Dataset)

Consider a corporate governance audit examining external audit costs across major public corporations (`Top500` table). Management wants to identify **which companies paid higher-than-average audit fees**, including their revenue, net income, and audit firm.

To construct this analysis, we write a scalar subquery that computes the population average audit fee:

```sql
SELECT 
    CompanyName, 
    Revenue, 
    Income, 
    AuditFees, 
    Auditor
FROM Top500
WHERE AuditFees > (
    SELECT AVG(AuditFees) 
    FROM Top500
)
ORDER BY AuditFees DESC
```

*Deconstructing the Execution Steps:*
1. **Inner Query Execution (First):** SQL evaluates `(SELECT AVG(AuditFees) FROM Top500)` across all 500 records. Suppose the overall average audit fee evaluates to **\$3,150,000**.
2. **Outer Query Execution (Second):** SQL substitutes the scalar result directly into the `WHERE` clause, effectively running:
   ```sql
   WHERE AuditFees > 3150000
   ```
3. **Filtering and Display:** SQL filters the table row-by-row, returning only those corporate clients whose audit fees exceed \$3.15M, sorted from highest to lowest.

*Output (Excerpt):*
| CompanyName | Revenue | Income | AuditFees | Auditor |
| :--- | :--- | :--- | :--- | :--- |
| Alcoa Corp | \$13,403,000,000 | \$227,000,000 | \$7,822,000 | PricewaterhouseCoopers LLP |
| American Airlines Group Inc | \$44,541,000,000 | \$1,412,000,000 | \$4,010,000 | KPMG LLP |
| Advance Auto Parts Inc | \$9,580,554,000 | \$423,847,000 | \$4,008,000 | Deloitte & Touche LLP |

### Additional Forensic Applications of Scalar Subqueries

Scalar subqueries are exceptionally versatile in accounting and risk management. Here are three common audit scenarios where scalar subqueries solve time-sequence and benchmark challenges:

#### 1. Dynamic Date Filtering (Relative to Latest Entry)

In real-world accounting databases, you rarely know the exact maximum date stored in a historical transaction table. Suppose an auditor wants to review all sales orders placed within the **last 30 days** of a database's recording history.

Instead of hardcoding a static date, use a scalar subquery to find the latest order date dynamically:

```sql
SELECT OrderID, OrderDate, CustomerID, Freight
FROM Orders
WHERE OrderDate >= DATEADD(day, -30, (
    SELECT MAX(OrderDate) 
    FROM Orders
))
ORDER BY OrderDate DESC
```
*Why this matters:* If the database is updated with new transaction files next month, this query automatically adjusts its 30-day window relative to the new maximum date without requiring code modifications!

#### 2. Low-Margin Product Exception Reporting

Inventory accountants routinely audit product profitability to identify items sold below portfolio contribution benchmarks. To flag products whose unit profit margin falls below the portfolio average:

```sql
SELECT 
    ProductID, 
    ProductName, 
    UnitPrice, 
    UnitCost,
    UnitPrice - UnitCost AS ContributionMargin
FROM Products
WHERE (UnitPrice - UnitCost) < (
    SELECT AVG(UnitPrice - UnitCost) 
    FROM Products
)
ORDER BY ContributionMargin ASC
```

---

## Summary and Advanced SQL Reference Checklist

Mastering `COUNT(DISTINCT)` and scalar subqueries unlocks sophisticated analytical workflows, allowing you to ask multi-layered questions of corporate databases.

### Key Chapter Takeaways

1. **`COUNT(DISTINCT)` Eliminates Duplicates:** Use `COUNT(DISTINCT Column)` to count unique entities (such as active customer accounts, unique vendor countries, or distinct product SKUs) rather than total raw transaction rows.
2. **Scalar Subqueries Solve Execution Precedence:** Because `WHERE` evaluates row-by-row before overall aggregations are computed, wrap summary queries in parentheses `()` to calculate benchmark values first.
3. **Dynamic Benchmarking:** Scalar subqueries eliminate hardcoded constants, ensuring your audit scripts automatically update when new transaction data is loaded into the system.

### Summary Comparison of Advanced SQL Retrieval Techniques

| Technique | SQL Syntax Example | What It Calculates | Primary Accounting / Auditing Role |
| :--- | :--- | :--- | :--- |
| **Row Count** | `COUNT(*)` | Total number of database records | Counting total journal entries or invoice volume. |
| **Non-Null Count** | `COUNT(Region)` | Number of populated entries | Identifying missing required fields or data incompleteness. |
| **Unique Values List** | `SELECT DISTINCT Country` | Multi-row table of unique values | Extracting unique vendor, customer, or country master lists. |
| **Unique Entity Count** | `COUNT(DISTINCT Country)` | Single scalar count of unique items | Quantifying geographic reach or client diversification. |
| **Scalar Subquery** | `WHERE Fee > (SELECT AVG(Fee)...)` | Filter rows against a dynamic benchmark | Detecting audit fee outliers, excessive expenses, or dynamic date windows. |